# Chaldene Animation Demo — Event-Driven Pipeline

This notebook shows how a **Python program** drives a Chaldene VP cell
through an animated sequence:

1. A Python cell computes 20 parameter frames for the pipeline.
2. It dispatches a `CustomEvent` to the JupyterLab frontend via `IPython.display.Javascript`.
3. A pre-registered JavaScript listener receives the event and steps through
   the frames at **one-second intervals**, calling `setInputValue` + `run` on each tick.

**Pipeline:** `read image → Gaussian Blur (sigma) → threshold (range)`

> Run cells in order with Shift+Enter.

In [ ]:
from PIL import Image
import numpy as np

arr = np.random.default_rng(42).integers(0, 256, (256, 256), dtype=np.uint8)
Image.fromarray(arr, mode='L').save('sample.png')
print('sample.png ready (256×256 grayscale)')


---
## Step 1 — The VP pipeline

Run this cell to open the visual canvas. The three-node pipeline will be
animated in Step 3: sigma ramps 0.5 → 5.25 while the threshold window narrows step by step.

In [ ]:
{"nodes":[{"id":"0","type":"read_image","position":{"x":80,"y":200},"selected":false,"data":{"specName":"read_image","displayLabel":"read image","description":"Reads a JPEG or PNG image from disk.","inputs":[{"id":"in0","name":"path","type":"string","displayLabel":"file","description":"Path to image.","defaultValue":"sample.png","widget":{"type":"FileInputFromServer","extensions":[".jpg",".jpeg",".png"]}},{"id":"in1","name":"mode","displayLabel":"mode","description":"Colour mode.","defaultValue":"GRAY","widget":{"type":"Dropdown","options":["GRAY","RGB"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image"}]}},{"id":"1","type":"read_image","position":{"x":460,"y":200},"selected":false,"data":{"specName":"Gaussian Blur","displayLabel":"gaussian denoise","description":"Apply Gaussian blur to remove noise.","inputs":[{"id":"in0","name":"image","type":"image","displayLabel":"image"},{"id":"in1","name":"sigma","displayLabel":"sigma","description":"Standard deviation for Gaussian kernel.","defaultValue":1.0,"widget":{"type":"Number","min":0,"step":0.1}},{"id":"in2","name":"mode","displayLabel":"mode","description":"Border handling mode.","defaultValue":"nearest","widget":{"type":"Dropdown","options":["reflect","constant","nearest","mirror","wrap"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image"}]}},{"id":"2","type":"read_image","position":{"x":840,"y":200},"selected":false,"data":{"specName":"threshold","displayLabel":"threshold","description":"Binarizes by keeping pixels within [lower, upper].","inputs":[{"id":"in0","name":"image","type":"image","displayLabel":"grayscale image","description":"Input image for thresholding."},{"id":"in1","name":"range","type":"tuple2","displayLabel":"Range","description":"Lower and upper threshold bounds.","defaultValue":[0.2,0.8],"widget":{"type":"HistogramRange","min":0,"max":1,"step":0.01}}],"outputs":[{"id":"out0","name":"image","type":"binary image","displayLabel":"image"}]}}],"edges":[{"id":"e01","source":"0","sourceHandle":"out0","target":"1","targetHandle":"in0","selected":false},{"id":"e12","source":"1","sourceHandle":"out0","target":"2","targetHandle":"in0","selected":false}]}

---
## Step 2 — Register the animation listener

This cell registers a `chaldene:start-animation` event handler on the browser
window. It stays idle until Step 3 fires the event.
Re-running this cell is safe — the old handler is removed first.

In [ ]:
%%javascript
const svc = window.__chaldene;
if (!svc) {
    element.innerHTML = '<b style="color:red">✗ Chaldene not found — install and reload.</b>';
} else {
    window.__svc = svc;
    window.__animCellId = svc.getReadyCellIds().slice(-1)[0];

    // Safe to re-run: remove the old handler before registering a new one
    if (window.__animListener) {
        window.removeEventListener('chaldene:start-animation', window.__animListener);
    }

    window.__animListener = (event) => {
        const { frames } = event.detail;
        const cellId = window.__animCellId;
        let step = 0;

        // Write to element ONCE at start and ONCE at end only.
        // Per-frame updates cause JupyterLab to scroll to this cell on every
        // tick, pulling the VP canvas out of view.
        element.innerHTML =
            '<b>Animation running — ' + frames.length + ' frames × 1 s</b><br>' +
            '<i style="color:#555">Watch the VP canvas in Step 1 while this plays.</i>';

        function tick() {
            if (step >= frames.length) {
                element.innerHTML =
                    '<b style="color:green">✓ Animation complete (' + frames.length + ' frames).</b>';
                return;
            }
            const f = frames[step];
            svc.setInputValue(cellId, '1', 'in1', f.sigma);
            svc.setInputValue(cellId, '2', 'in1', f.range);
            svc.run(cellId);
            step++;
            setTimeout(tick, 1000);
        }

        tick();
    };

    window.addEventListener('chaldene:start-animation', window.__animListener);
    element.innerHTML = '<b style="color:#555">✓ Listener ready.</b> Run Step 3 to start.';
}


---
## Step 3 — Run the Python program

This is the **Jupyter program**: it builds the 20 parameter frames, prints
a summary table, then fires `chaldene:start-animation` with the data.
The listener picks it up and animates the VP cell one frame per second.

In [ ]:
import json
from IPython.display import Javascript, display

# 20 frames: sigma ramps 0.5 → 5.25 while threshold window narrows
frames = [
    {
        'sigma': round(0.5 + i * 0.25, 2),
        'range': [round(0.05 + i * 0.04, 2), round(0.95 - i * 0.02, 2)]
    }
    for i in range(20)
]

for i, f in enumerate(frames):
    print(f'frame {i+1:2d}: sigma={f["sigma"]:.2f}  range={f["range"]}')

display(Javascript(f"""
window.dispatchEvent(new CustomEvent('chaldene:start-animation', {{
    detail: {{ frames: {json.dumps(frames)} }}
}}));
"""))


---
## How it works

```
Python cell                  JupyterLab frontend
────────────────────             ────────────────────────────────────────
build frames[]               VP cell rendered (Step 1)
   │                            │
display(Javascript)  ─────►  CustomEvent('chaldene:start-animation')
                                │
                             listener fires, then for each frame (1 s):
                               setInputValue(cellId, '1', 'in1', sigma)
                               setInputValue(cellId, '2', 'in1', range)
                               run(cellId)
```

`display(Javascript(...))` is the bridge: it serialises the Python data as
JSON and injects it as a DOM event. The JS side never polls — it just reacts.

To adapt this pattern: replace the frame list with any Python computation
(optimiser output, sensor readings, grid-search results) and adjust the
event name and VP cell parameters.